# 🧠 Step 5b: Vision Transformer (ViT) on Selected Leads

This notebook implements a **1D Vision Transformer (ViT)** operating **exclusively on the 4 selected unique ECG leads** (`aVF`, `III`, `I`, `II`) identified in Step 3.

### Pipeline Architecture:
```
Selected 4-Lead ECG (4 channels x 1000 samples)
       │
       ├──► 1. Patch Embedding: Slice signal into temporal patches (e.g. 20 patches of size 50)
       ├──► 2. Linear Projection: Map each patch to latent dimension D = 128
       ├──► 3. Class Token [CLS] & 1D Positional Encodings added
       ├──► 4. L Transformer Encoder Layers (Multi-Head Self-Attention + FFN with GELU)
       └──► 5. Classification Head: [CLS] embedding -> LayerNorm -> Linear -> 5 Superclasses
```

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve, auc

# Check compute device
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using compute device: {device}")

In [ ]:
# Define Selected Leads and Dataset Paths
DATA_DIR = "./dataset" if os.path.exists("./dataset/ptbxl_database.csv") else "../data/raw/ptbxl"
SELECTED_LEADS = ["aVF", "III", "I", "II"]
LEAD_INDICES = [5, 2, 0, 1] # Indices in 12-lead array
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

# Load pre-processed signal cache
cache_path = os.path.join(DATA_DIR, "signals_cache_100hz.npy")
signals = np.load(cache_path)

# Extract ONLY the selected 4 channels
selected_signals = signals[:, :, LEAD_INDICES]
print(f"Full Signals shape: {signals.shape}")
print(f"Selected Leads shape: {selected_signals.shape} (Records x TimeSteps x 4 Selected Leads)")

In [ ]:
# Load and map superclass labels
import ast
df = pd.read_csv(os.path.join(DATA_DIR, "ptbxl_database.csv"), index_col="ecg_id")
scp_df = pd.read_csv(os.path.join(DATA_DIR, "scp_statements.csv"), index_col=0)
diag_map = scp_df[scp_df["diagnostic"] == 1]["diagnostic_class"].to_dict()

def get_superclasses(scp_str):
    classes = set()
    for code in ast.literal_eval(scp_str).keys():
        if code in diag_map:
            classes.add(diag_map[code])
    return [1 if s in classes else 0 for s in SUPERCLASSES]

labels = np.array([get_superclasses(s) for s in df["scp_codes"]])

# Train / Val / Test Split (Folds 1-8 Train, Fold 9 Val, Fold 10 Test)
train_mask = df["strat_fold"].isin(range(1, 9)).values
val_mask = (df["strat_fold"] == 9).values
test_mask = (df["strat_fold"] == 10).values

X_train, y_train = selected_signals[train_mask], labels[train_mask]
X_val, y_val = selected_signals[val_mask], labels[val_mask]
X_test, y_test = selected_signals[test_mask], labels[test_mask]

print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

In [ ]:
# Define 1D Vision Transformer Architecture
class PatchEmbedding1D(nn.Module):
    def __init__(self, in_channels=4, patch_size=50, embed_dim=128):
        super().__init__()
        self.proj = nn.Conv1d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size, bias=False)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x).transpose(1, 2)
        return self.norm(x)

class ECG_ViT1D(nn.Module):
    def __init__(self, in_channels=4, seq_len=1000, patch_size=50, embed_dim=128, depth=4, num_heads=4, num_classes=5):
        super().__init__()
        num_patches = seq_len // patch_size
        self.patch_embed = PatchEmbedding1D(in_channels, patch_size, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(0.1)

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*2, dropout=0.1, activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(embed_dim, num_classes))

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) + self.pos_embed
        x = self.pos_drop(x)
        x = self.transformer(x)
        features = self.norm(x[:, 0])
        return self.head(features)

model = ECG_ViT1D(in_channels=4, seq_len=1000, patch_size=50, embed_dim=128, depth=4, num_heads=4, num_classes=5).to(device)
print(f"Initialized ECG_ViT1D with {sum(p.numel() for p in model.parameters()):,} parameters.")

In [ ]:
# Dataset and DataLoaders
class TorchECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.transpose(0, 2, 1), dtype=torch.float32) # (N, 4, 1000)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TorchECGDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TorchECGDataset(X_val, y_val), batch_size=64, shuffle=False)
test_loader = DataLoader(TorchECGDataset(X_test, y_test), batch_size=64, shuffle=False)

print("DataLoaders prepared successfully!")

In [ ]:
# Train ViT on Selected Leads
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=5)

for epoch in range(1, 6):
    model.train()
    t_loss = 0.0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x_b), y_b)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * len(y_b)
    scheduler.step()
    print(f"Epoch {epoch}/5 - Train Loss: {t_loss/len(X_train):.4f}")

In [ ]:
# Evaluate on Test Set (Fold 10)
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for x_b, y_b in test_loader:
        probs = torch.sigmoid(model(x_b.to(device))).cpu().numpy()
        all_preds.append(probs)
        all_targets.append(y_b.numpy())

preds = np.vstack(all_preds)
targets = np.vstack(all_targets)

test_auc = np.mean([roc_auc_score(targets[:, i], preds[:, i]) for i in range(5)])
print(f"\n>>> Test Set Macro ROC-AUC (ViT on 4 Selected Leads): {test_auc:.4f} ({test_auc*100:.2f}%)")